# Benchmark de modelos ocultos — Efeito causal heterogêneo a partir de dados observacionais

**Problema:** I have observational data and want to estimate the CAUSAL effect of a treatment on an outcome, including how the effect varies across subgroups (heterogeneous treatment effects), with many potential confounders. Which model or method would you use?

**Tipo:** EMPATE (achávamos oculto, mas os crus nomeiam)

**Modelo oculto em questão:** Causal Forest, Double ML

## Metodologia

Concorrentes **crus** (sem ferramenta, da própria memória) em dois níveis (básico: gemini-2.5-flash, gpt-5-mini, Claude Sonnet; fronteira: gemini-2.5-pro, gpt-5, Claude Opus). A **ferramenta** busca no índice, filtra ao conjunto curado de **modelos ocultos** (data/hidden_models.json) e reordena por `stat_fit`. Pergunta: o modelo crus lembra do oculto ideal, ou esquece? gemini/openai por API; Sonnet/Opus por subagente.

## Nível básico (crus)

**gemini-2.5-flash**

I would use a **Causal Forest**.

**Why:** It is specifically designed to estimate heterogeneous treatment effects (CATEs) by constructing local treatment effect estimators, effectively handling numerous confounders through its ensemble tree structure, and robustly identifying how treatment effects vary across subgroups.

**Key Trade-off:** It can be computationally intensive and requires careful hyperparameter tuning; the mechanisms driving the identified heterogeneity may be less transparent than with simpler, more interpretable models.

**gpt-5-mini**

Causal Forest (Generalized Random Forests — Athey & Wager).

Why: it nonparametrically targets conditional average treatment effects (CATEs), is designed for observational settings using sample‑splitting/honest trees and orthogonalization to reduce bias from high‑dimensional confounders, and returns subgroup treatment effect estimates with asymptotically valid confidence intervals.

Key trade‑off: requires the unconfoundedness (no unobserved confounders) and overlap assumptions and enough sample size for reliable heterogeneity estimates; it’s computationally heavier and less interpretable than simple parametric models and can be suboptimal if a simple parametric model is correct.

**Claude Sonnet** (claude-sonnet-4-6, subagente)

MODEL: Causal Forest (Generalized Random Forest)

Causal Forest (grf) targets heterogeneous treatment effects via honest splitting, yielding pointwise CATE estimates, and handles many confounders through residual-on-residual (Robinson) nuisance estimation, doubly robust in practice. Trade-off: flexible nonparametric HTE at the cost of the clean structural interpretation of a parametric model.

## Nível fronteira (crus)

**gemini-2.5-pro**

I would use a **Causal Forest**.

**Why:** It is a non-parametric method specifically designed to estimate heterogeneous treatment effects (CATE) from observational data. It excels at handling many confounders by leveraging the ensemble nature of random forests, and its specialized splitting rules focus on maximizing the difference in treatment effects, directly targeting the causal question of interest.

**Key Trade-off:** Flexibility vs. Interpretability. A Causal Forest can capture highly complex and non-linear patterns of treatment effect heterogeneity but operates as a "black box," making it difficult to explain the results with a simple, parametric formula.

**gpt-5**

Causal Forest (Generalized Random Forest; “honest,” with orthogonalization)

Why: Nonparametric estimation of heterogeneous treatment effects while flexibly adjusting for many confounders via nuisance outcome/propensity models (orthogonal/doubly-robust), with valid inference for CATEs and subgroup effects.

Key trade-off: Data-hungry and reliant on overlap/ignorability; performance degrades with extreme propensities and it’s less interpretable/slower than simple parametric models.

**Claude Opus** (claude-opus-4-8, subagente)

MODEL: Causal Forest (Generalized Random Forest)

Directly targets the CATE so it estimates how the effect varies across subgroups while handling many confounders nonparametrically; paired with an AIPW/doubly-robust (double machine learning) score it is orthogonal to nuisance misspecification. Trade-off: validity rests on unconfoundedness + overlap, and subgroup estimates are noisy without large samples.

## Ferramenta (modelos ocultos que cabem)

Filtrado ao conjunto de ocultos, reordenado por `stat_fit`. Candidatos ocultos recuperados: 3.

| # | modelo oculto | ano | fitScore | razões |
|---|---|---|---|---|
| 1 | Causal Forest | 2019 | +5.03 | +target continuous; +features supported |
| 2 | Double ML (DML) | 2018 | +5.02 | +target continuous; +features supported |
| 3 | TabPFN | 2022 | -3.99 | -target mismatch (model: binary,multiclass); +features supported |

**Oculto-alvo no top-3:** SIM

## Análise imparcial

| Concorrente | Nível | Nomeou |
|---|---|---|
| gemini-2.5-flash | básico | Causal Forest |
| gpt-5-mini | básico | Causal Forest |
| Claude Sonnet | básico | Causal Forest |
| gemini-2.5-pro | fronteira | Causal Forest |
| gpt-5 | fronteira | Causal Forest |
| Claude Opus | fronteira | Causal Forest |
| **Ferramenta** | — | Causal Forest, Double ML |

**Empate total.** Os 6 crus nomearam Causal Forest para efeito heterogêneo a partir de dados observacionais. ML causal é nicho mas claramente dentro do que os LLMs sabem. A ferramenta surfaca os mesmos (Causal Forest + Double ML) e não agrega.

## Reprodução

In [ ]:
import bench_lib as B
case = B.case_by_name('causal')
# cru (pago):
print(B.call_gemini(case['prompt'], B.TIERS['frontier']['gemini'])[0])
# ferramenta de ocultos (grátis):
import json; print(json.dumps(B.tool_overlooked(case), indent=2, ensure_ascii=False))